# AnomalyCLIP on Real-IAD — running directly on LUMI
Phase 1: evaluate the VisA checkpoint on a held-out Real-IAD split. Phase 2: fine-tune on Real-IAD, re-evaluate, compare.
Run this notebook on LUMI; jobs are submitted with `sbatch` from `scripts/`. See README.

In [ ]:
import os, re, subprocess
from pathlib import Path
import pandas as pd

ACCOUNT  = os.environ.get("LUMI_ACCOUNT", "project_465000000")
WORKDIR  = Path(os.environ.get("LUMI_WORKDIR", f"/scratch/{ACCOUNT}/{os.environ['USER']}/anomalyclip_demo"))
SCRIPTS  = Path("scripts").resolve()

REPO         = WORKDIR / "AnomalyCLIP"
VISA_CKPT    = REPO / "checkpoints/9_12_4_multiscale_visa/epoch_15.pth"
REALIAD_ROOT = WORKDIR / "realiad_1024"
VENV         = WORKDIR / "venv"
DATA_EVAL    = WORKDIR / "data/eval"
DATA_FT      = WORKDIR / "data/ft"
FT_CKPT_DIR  = WORKDIR / "ckpt_finetuned"
RESULTS_DIR  = WORKDIR / "results"
FT_EPOCHS, FT_LR = 5, 1e-4

LOGS = WORKDIR / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

## Helpers

In [ ]:
def run_job(script: str, name: str, **env) -> str:
    """sbatch scripts/<script> with env vars, wait for it to finish, return the job id."""
    env = {"SCRIPTS_DIR": SCRIPTS, "VENV": VENV, **env}
    export = "ALL," + ",".join(f"{k}={v}" for k, v in env.items())
    out = subprocess.run(["sbatch", "--parsable", "--wait", "-A", ACCOUNT, f"--export={export}",
                          str(SCRIPTS / script)], cwd=WORKDIR, capture_output=True, text=True)
    jid = out.stdout.strip().split(";")[0]
    print(f"[{name}] job {jid} finished, exit code {out.returncode}")
    if out.returncode != 0:
        raise RuntimeError(f"{name} failed: {out.stderr}\nsee {LOGS}")
    return jid

def read_log(prefix: str, jid: str) -> str:
    return (LOGS / f"{prefix}-{jid}.out").read_text()

def parse_metrics(log: str) -> dict:
    """Parse the mean row of AnomalyCLIP's result table using its header names."""
    lines = [l for l in log.splitlines() if l.strip()]
    hdr = next(i for i in range(len(lines) - 1, -1, -1) if "pixel_auroc" in lines[i])
    names = [n for n in re.findall(r"[a-z_]+", lines[hdr]) if n != "objects"]
    mean = next(l for l in lines[hdr + 1:] if re.match(r"^\W*mean\b", l))
    vals = [float(v) for v in re.findall(r"-?\d+\.\d+|-?\d+", mean.split("mean", 1)[1])]
    return dict(zip(names, vals))

_demo = "| objects | pixel_auroc | pixel_aupro | image_auroc | image_ap |\n| a | 90.1 | 80.2 | 70.3 | 60.4 |\n| mean | 91.5 | 81.5 | 71.5 | 61.5 |"
assert parse_metrics(_demo)["image_auroc"] == 71.5

## 1. Download Real-IAD (gated: needs `HF_TOKEN` in the environment)

In [ ]:
run_job("download_data.sbatch", "download", REALIAD_ROOT=REALIAD_ROOT, HF_TOKEN=os.environ["HF_TOKEN"])

## 2. Build the eval / fine-tune splits (seeded 50/50 split of Real-IAD labelled test images)

In [ ]:
run_job("convert_data.sbatch", "data", REALIAD_ROOT=REALIAD_ROOT, DATA_EVAL=DATA_EVAL, DATA_FT=DATA_FT)

## 3. Phase 1: zero-shot evaluation with the VisA checkpoint

In [ ]:
eval_env = dict(REPO=REPO, DATA=DATA_EVAL, RESULTS_DIR=RESULTS_DIR)
jid = run_job("eval.sbatch", "eval-visa", CKPT=VISA_CKPT, TAG="eval-visa", **eval_env)
before = parse_metrics(read_log("eval", jid))
before

## 4. Phase 2: fine-tune on Real-IAD and re-evaluate

In [ ]:
run_job("finetune.sbatch", "finetune", REPO=REPO, DATA=DATA_FT, OUT=FT_CKPT_DIR, EPOCHS=FT_EPOCHS, LR=FT_LR)
jid = run_job("eval.sbatch", "eval-finetuned", CKPT=FT_CKPT_DIR / "finetuned.pth", TAG="eval-finetuned", **eval_env)
after = parse_metrics(read_log("eval", jid))
after

## 5. Before vs. after

In [ ]:
df = pd.DataFrame({"VisA ckpt (zero-shot)": before, "Fine-tuned on Real-IAD": after})
df["delta"] = df.iloc[:, 1] - df.iloc[:, 0]
df.round(2)